In [ ]:
# Parameters -- Fabric overrides at runtime.
tenant_id      = 100          # the practice just seeded by step 2
warehouse_name = "WH_Dentally"
build_timeout  = 14400        # s -- Bronze..Gold + model refresh
refresh_model  = True         # refresh the semantic model at the end


In [ ]:
import struct
import pyodbc
import sempy.fabric as fabric

_ws_id = fabric.get_workspace_id()
_whs   = fabric.FabricRestClient().get("/v1/workspaces/" + _ws_id + "/warehouses").json()["value"]
_wh    = next((w for w in _whs if w["displayName"] == warehouse_name), None)
if _wh is None:
    raise RuntimeError("Warehouse '" + warehouse_name + "' not found in workspace " + _ws_id)
endpoint = _wh["properties"]["connectionString"]

_tok    = mssparkutils.credentials.getToken("https://database.windows.net/")
_tb     = _tok.encode("UTF-16-LE")
_struct = struct.pack("<I" + str(len(_tb)) + "s", len(_tb), _tb)
conn = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};Server=" + endpoint + ",1433;Database=" + warehouse_name
    + ";Encrypt=yes;TrustServerCertificate=no;", attrs_before={1256: _struct})
conn.autocommit = True
cur = conn.cursor()
print("ONBOARDING step 3 of 3 -- tenant " + str(tenant_id))
print("Connected to " + warehouse_name + " @ " + endpoint)


In [ ]:
print("\nCHECK  the tenant has been seeded (step 2) and the sync has landed it")
cur.execute("SELECT (SELECT COUNT(*) FROM Input.Targets WHERE Tenant_ID = ?), "
            "       (SELECT COUNT(*) FROM Input.Practitioner_Role WHERE Tenant_ID = ?), "
            "       (SELECT COUNT(*) FROM Input.Plan_Capitation_Rate WHERE Tenant_ID = ?)",
            tenant_id, tenant_id, tenant_id)
_t, _r, _p = cur.fetchone()
print("  Input.Targets=" + str(_t) + "  Practitioner_Role=" + str(_r)
      + "  Plan_Capitation_Rate=" + str(_p))
if _t == 0 and _r == 0:
    raise RuntimeError(
        "Tenant " + str(tenant_id) + " has no seeded metadata in the WAREHOUSE Input.* tables. "
        "Either step 2 has not run (cd Web; python onboarding_seed.py --tenant "
        + str(tenant_id) + " --apply) or it ran but was never synced across "
        "(python appdb_sync.py full). Republishing now would change nothing.")
print("  seeded -- proceeding.")


In [ ]:
print("\nSTEP 7  REBUILD Bronze..Gold so every fact picks up the seeded Input.*")
# Ingests OFF -- step 1 already pulled, and Stage still holds it. The parameter names matter:
# run_dentally_ingest / run_xero_ingest are what Orchestrate_Build actually reads. A name it
# does not define is injected as a harmless new variable and changes nothing, so a typo here
# would quietly run a full Dentally pull a second time.
mssparkutils.notebook.run("Orchestrate_Build", build_timeout, {
    "full_refresh":           False,
    "run_dentally_ingest":    False,
    "run_xero_ingest":        False,
    "refresh_semantic_model": refresh_model,
})
print("  rebuild complete" + (" (model refreshed)" if refresh_model else " (model NOT refreshed)"))


In [ ]:
print("\n" + "=" * 70)
print("ONBOARDING COMPLETE for tenant " + str(tenant_id) + ".")
print("=" * 70)
print("  Open the reports as the customer would and check they look ALIVE:")
print("    - the Day Book shows targets, not blanks")
print("    - plan capitation revenue is non-zero")
print("    - practitioner profitability is populated")
print("  If any of those are still empty, step 2 did not land -- check")
print("  Input.Targets / Input.Plan_Capitation_Rate for this tenant.")
print("")
print("  ==> TELL THEM THE METADATA IS ESTIMATED. <== Roles, FTE, associate % and plan")
print("  rates were DERIVED from their Dentally data, not supplied by them. They are")
print("  close enough to make the reports work and wrong in the details -- the plan")
print("  rates especially, which are all seeded at one flat monthly figure.")
print("  Settings is where they correct it, and nothing they change is overwritten.")
print("=" * 70)
conn.close()
